In [2]:
import numpy as np
import pymc3 as pm
import scipy as sp

In [7]:
poiss_lam = 300.0
exp_scale = 0.5
sample_size = 100000

least_prob = 1.0

dev_val_vec = []
dur_val_vec = []
tot_prob_vec = []
rel_score_vec = []

for i in range(sample_size):
    dev_val = np.random.exponential(scale=exp_scale)
    dur_val = np.random.poisson(lam=poiss_lam)
    dev_val_vec.append(dev_val)
    dur_val_vec.append(dur_val)
    
    dur_prob = 0.1  # (poiss_lam ** dur_val)*np.exp(-poiss_lam)/sp.special.factorial(dur_val)
    dev_prob = 0.1  # (1/exp_scale)*np.exp(-dev_val/exp_scale)

    tot_prob = dev_prob*dur_prob
    
    tot_prob_vec.append(tot_prob)
    
    if tot_prob < least_prob:
        least_prob = tot_prob

for tot_prob in tot_prob_vec:
    rel_score_vec.append(100*(least_prob/tot_prob))

dev_model = pm.Model()
dur_model = pm.Model()

with dev_model:
    lambda_est = pm.Uniform('lambda_est', lower=0, upper=20)
    y_exp = pm.Exponential('y_exp', lam=lambda_est, observed=dev_val_vec)

with dur_model:
    mu_est = pm.Uniform('mu_est', lower=0, upper=20)
    y_poiss = pm.Poisson('y_poiss', mu=mu_est, observed=dur_val_vec)

with dev_model:
    trace = pm.sample(2000, cores=1, chains=2, tune=1000)
    lambda_est_val = np.mean(trace['lambda_est'])
    
with dur_model:
    trace = pm.sample(2000, cores=1, chains=2, tune=1000)
    mu_est_val = np.mean(trace['mu_est'])

print("Estimated Lambda for deviation = {}, estimated mu for duration = {}".format(lambda_est_val, mu_est_val))

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [lambda_est]
Sampling chain 1, 0 divergences: 100%|██████████| 3000/3000 [00:11<00:00, 252.86it/s]
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [mu_est]
Sampling chain 1, 0 divergences: 100%|██████████| 3000/3000 [00:20<00:00, 145.56it/s]
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


Estimated Lambda for deviation = 1.9904939156966237, estimated mu for duration = 19.99999930343336


In [8]:
print("Maximum deviation {}".format(max(dev_val_vec)))
print("Minimum deviation {}".format(min(dev_val_vec)))

print("Maximum duration {}".format(max(dur_val_vec)))
print("Minimum duration {}".format(min(dur_val_vec)))

Maximum deviation 5.63244218496226
Minimum deviation 4.729643874344852e-06
Maximum duration 381
Minimum duration 227
